<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/22-research-practice-emerging-frontiers.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **研究实践与新兴前沿** {#research-practice-emerging-frontiers}

深度学习研究并不是先制造一个更大的模型，再附上一项看起来有利的分数。它是一套严格的过程：把重要的不确定性转化为可证伪的问题，构造能够区分多种合理解释的证据，并明确结论究竟可以推广到多远。因此，完整的研究产物不仅是最终检查点，还包括数据沿袭、代码、运行环境、被否定的假设、资源预算与失败分析。

本章前半部分建立这套过程，后半部分再用它审视新兴方向，而不是把每项近期结果都当作持久范式。当某个方向的机制与评估协议已在多项独立工作中积累时，本文称其为**已形成稳定基础**；当有前景的结果与重大未解约束并存时，称其为**活跃研究方向**；当核心假设或部署路径仍缺乏充分检验时，则称其为**推测性方向**。

代码围绕 scikit-learn 所提供的 [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) 数据集副本展开，该数据集采用 **CC BY 4.0** 许可。全章研究同一个紧凑问题：*在固定训练预算下，隐藏层宽度与 dropout 会如何影响干净样本准确率、损坏输入准确率和不确定性？* 确定性的训练/验证/测试划分会被基线、消融、缩放先导实验、稀疏计算、测试时计算、持续学习、因果偏移演示和约束解码共同复用。这些实验用于教授研究方法；一个 8×8 手写数字基准并不能证明前沿规模模型也具有相同行为。

![研究周期把规划、收集、处理、分析、发布、保存和复用连接起来。](assets/dl22-research-cycle.jpg){fig-align="center" width="76%" fig-alt="一幅手绘循环图把研究数据规划、收集、处理、分析、发布、保存和复用连接起来。"}

*来源：The Turing Way Community 与 Scriberia，[Research Cycle](https://doi.org/10.5281/zenodo.3332807)，CC BY 4.0。该循环强调发布并不是终点；被妥善保存的研究产物能够支持审查与复用。*

<details>
<summary><strong>PyTorch：建立全章共享的研究工作负载</strong></summary>

```python
import hashlib
import json
import platform
import random
import time

import numpy as np
import sklearn
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=2201):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_ids = np.arange(len(digits.data))
train_ids, heldout_ids = train_test_split(
    all_ids, test_size=0.40, stratify=digits.target, random_state=2201
)
val_ids, test_ids = train_test_split(
    heldout_ids,
    test_size=0.50,
    stratify=digits.target[heldout_ids],
    random_state=2201,
)

# UCI documents pixel intensities on a fixed 0..16 scale, so this transform
# does not estimate a statistic from validation or test examples.
images = torch.tensor(digits.images / 16.0, dtype=torch.float32).flatten(1)
targets = torch.tensor(digits.target, dtype=torch.long)


def make_loader(ids, batch_size=64, shuffle=False, seed=2201):
    dataset = TensorDataset(images[ids], targets[ids])
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


class DigitMLP(nn.Module):
    def __init__(self, width=64, dropout=0.10):
        super().__init__()
        self.width = width
        self.fc1 = nn.Linear(64, width)
        self.fc2 = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(width, 10)

    def features(self, x):
        h1 = F.gelu(self.fc1(x))
        h2 = F.gelu(self.fc2(self.dropout(h1)))
        return h1, h2

    def forward(self, x):
        return self.out(self.features(x)[1])


def evaluate(model, ids, noise_std=0.0, seed=0):
    model.eval()
    x = images[ids]
    if noise_std:
        generator = torch.Generator().manual_seed(seed)
        noise = torch.randn(x.shape, generator=generator) * noise_std
        x = (x + noise).clamp(0.0, 1.0)
    with torch.inference_mode():
        logits = model(x)
        probabilities = logits.softmax(dim=1)
    accuracy = float(logits.argmax(1).eq(targets[ids]).float().mean())
    nll = float(F.cross_entropy(logits, targets[ids]))
    return {"accuracy": accuracy, "nll": nll, "probabilities": probabilities, "logits": logits}


def fit_model(seed=2201, width=64, dropout=0.10, epochs=12, lr=2e-3):
    seed_everything(seed)
    model = DigitMLP(width=width, dropout=dropout)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = make_loader(train_ids, shuffle=True, seed=seed)
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    clean = evaluate(model, test_ids)
    corrupt = evaluate(model, test_ids, noise_std=0.22, seed=999)
    return model, {
        "clean_accuracy": clean["accuracy"],
        "corrupt_accuracy": corrupt["accuracy"],
        "clean_nll": clean["nll"],
    }


baseline_model, baseline_metrics = fit_model()
assert len(set(train_ids) & set(test_ids)) == 0
assert baseline_metrics["clean_accuracy"] > 0.88
print({k: round(v, 3) for k, v in baseline_metrics.items()})
```

</details>

这里把单幅图像作为划分单位，因为当前封装的 UCI Digits 副本没有提供受试者标识。在医疗、用户、说话人、家庭或时间相关研究中，按单行随机划分经常会泄漏身份或未来信息。划分边界必须从数据生成过程推导，而不能由某个方便调用的划分函数决定。


### **阅读并梳理研究论文** {#reading-mapping-research-paper}

阅读论文时，首先应把它看作一个论证。摘要宣传结果，但真正的科学对象是一条由问题、所提机制、证据、假设和受限贡献组成的链条。只按文档顺序阅读，很容易记住架构细节，却忽视比较实验是否真正回答了论文声称的问题。

![研究论文可以被梳理为从问题到受限贡献的论证链。](assets/dl22-paper-claim-map.svg){fig-align="center" width="76%" fig-alt="五个方框依次连接研究问题、机制、证据、边界和贡献。"}

首先制作一张**主张图**。用一句话写出主要主张，并包含研究总体、干预、比较对象、指标和预算。然后确定哪种机制应当导致观察到的增益。面对每张结果表，都要询问它排除了哪种替代解释：更多参数、更多数据、不同划分、更长调参、偏向某种方法的预处理，还是选择性汇报。最后提取该主张尚未被检验的条件。一个基准和一个随机种子上的结果只是一次观察，还不是普遍规律。

三遍阅读通常比从头到尾逐行阅读更高效：

1. **初筛：**检查研究问题、结果表、局限、数据和计算预算。判断论文是否相关，以及现有证据能否支撑宣传的适用范围。
2. **机制：**追踪张量形状、目标函数、算法步骤和计算成本。重新推导能够把新方法与基线区分开的最小公式。
3. **审计：**重建数据划分、调参预算、不确定性、消融、发布产物与可能的混杂因素。[NeurIPS paper checklist](https://neurips.cc/public/guides/PaperChecklist) 很有帮助，因为它把关于假设、可复现性、资源和社会影响的汇报要求转化为具体问题。

<details>
<summary><strong>Python：把主张图编码成能够验证失败的结构</strong></summary>

```python
claim_card = {
    "question": "Do width and dropout improve corrupted-digit accuracy at fixed epochs?",
    "population": "UCI Digits under synthetic Gaussian corruption",
    "intervention": {"width": [64, 128], "dropout": [0.0, 0.2]},
    "comparator": "width=64, dropout=0.0",
    "primary_metric": "corrupt_accuracy",
    "secondary_metrics": ["clean_accuracy", "clean_nll"],
    "fixed_budget": {"epochs": 8, "seeds": [2201, 2202, 2203]},
    "boundary": "mechanism study on 8x8 digits, not a scale-law benchmark",
}

required = {
    "question",
    "population",
    "intervention",
    "comparator",
    "primary_metric",
    "fixed_budget",
    "boundary",
}
assert required <= claim_card.keys()
assert claim_card["primary_metric"] not in claim_card["secondary_metrics"]
print(json.dumps(claim_card, indent=2))
```

</details>

这张卡片刻意比普通文字摘要更严格。如果无法填写比较对象、预算或研究总体，读者就找到了一个会影响解释或复现的重要歧义。引用量和基准排名都无法修复这类歧义。


### **复现、重复验证与基线** {#reproduction-replication-baselines}

不同领域对术语的使用并不完全一致，因此项目应给出可操作定义。本文中，**可重复运行（repeatability）**指在相同代码和环境中再次运行；**计算复现（computational reproduction）**指利用已发布产物重新生成论文所报告的结果；**独立重复验证（replication）**指使用独立构建的实现或数据集检验同一个科学主张；**稳健性分析（robustness analysis）**则改变合理的干扰条件，例如随机种子、划分、预算或领域。确定性重跑很有价值，但它检验的命题远窄于独立重复验证。

![可重复运行、计算复现、独立重复验证、稳健性和决策范围构成逐步增强的证据。](assets/dl22-reproducibility-chain.svg){fig-align="center" width="76%" fig-alt="五阶段证据链从可重复运行推进到有边界的决策，下面配有基线阶梯。"}

基线是控制条件，而不是陪衬性质的弱模型。有用的基线阶梯包括：用于检测泄漏的 sanity baseline、用于衡量任务难度的简单方法、在数据与调参预算上匹配的强方法，以及只有在额外信息合理时才成立的上界。比较必须固定那些不属于研究问题的资源。如果新方法获得了更多 token、增强、超参数试验或测试时调用，这些都是干预因素，必须报告。

可复现性要求识别影响结果的完整状态：数据集版本和校验和、划分索引、预处理、软件包版本、随机种子、模型配置、优化器、更新次数、与硬件相关的内核以及评估代码。逐比特一致并非总能实现，也不总是科学上必需；目标可以是在事先声明的数值或统计容差内达成一致。

<details>
<summary><strong>Python：验证确定性重跑并创建最小清单</strong></summary>

```python
def state_digest(model):
    payload = b"".join(
        tensor.detach().cpu().contiguous().numpy().tobytes()
        for tensor in model.state_dict().values()
    )
    return hashlib.sha256(payload).hexdigest()


repeat_a, metrics_a = fit_model(seed=2207, epochs=6)
repeat_b, metrics_b = fit_model(seed=2207, epochs=6)
assert state_digest(repeat_a) == state_digest(repeat_b)
assert metrics_a == metrics_b

dataset_digest = hashlib.sha256(
    np.ascontiguousarray(digits.data).tobytes()
    + np.ascontiguousarray(digits.target).tobytes()
).hexdigest()
manifest = {
    "dataset": "UCI Optical Recognition of Handwritten Digits",
    "dataset_sha256": dataset_digest,
    "split_sha256": hashlib.sha256(
        np.concatenate([train_ids, val_ids, test_ids]).astype(np.int64).tobytes()
    ).hexdigest(),
    "seed": 2207,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "sklearn": sklearn.__version__,
    "model_sha256": state_digest(repeat_a),
}
assert len(manifest["dataset_sha256"]) == 64
print({"repeatable": True, "clean_accuracy": round(metrics_a["clean_accuracy"], 3)})
```

</details>

该测试证明当前 CPU 软件栈上的可重复运行，但没有证明独立重复验证，也不应推广到不同的加速器内核。发布内容应说明自己承诺的是哪一级可复现性，并提供可以重新生成每张结果表或图的命令。


### **消融与受控实验** {#ablations-controlled-experiments}

消融实验用于询问哪一个组件造成了观察到的差异。只有在训练预算、数据、参数统计方式、调参策略与评估均保持一致时，从最终模型中删除某个组件才具有解释力；否则实验会同时改变多个原因。

对于宽度 (A) 与 dropout (B) 两个二元因素，完整的 (2\times2) 设计可以同时估计主效应与交互效应。若 (m_{ab}) 表示因素水平 (a,b\in\{0,1\}) 下的平均指标，则交互项为

$$
I_{AB}=(m_{11}-m_{10})-(m_{01}-m_{00}).
$$

非零 (I_{AB}) 表示 dropout 的作用依赖于宽度。只汇报参考模型和组合模型，无法区分两个可相加的收益、真正的交互，或者一个有害组件被另一个组件掩盖的情况。

![二乘二消融设计可以识别主效应与交互效应。](assets/dl22-ablation-matrix.svg){fig-align="center" width="72%" fig-alt="一个二乘二实验矩阵交叉比较隐藏层宽度和 dropout，并展示交互对比。"}

<details>
<summary><strong>PyTorch：运行多随机种子的析因消融实验</strong></summary>

```python
ablation_rows = []
ablation_models = {}
for width in (64, 128):
    for dropout in (0.0, 0.2):
        for seed in (2201, 2202, 2203):
            candidate, metrics = fit_model(
                seed=seed, width=width, dropout=dropout, epochs=8
            )
            ablation_models[(width, dropout, seed)] = candidate
            ablation_rows.append(
                {"width": width, "dropout": dropout, "seed": seed, **metrics}
            )


def grouped_mean(metric):
    return {
        (width, dropout): float(
            np.mean(
                [
                    row[metric]
                    for row in ablation_rows
                    if row["width"] == width and row["dropout"] == dropout
                ]
            )
        )
        for width in (64, 128)
        for dropout in (0.0, 0.2)
    }


corrupt_means = grouped_mean("corrupt_accuracy")
interaction = (
    corrupt_means[(128, 0.2)]
    - corrupt_means[(128, 0.0)]
    - corrupt_means[(64, 0.2)]
    + corrupt_means[(64, 0.0)]
)
assert len(ablation_rows) == 12
print({
    "means": {str(k): round(v, 3) for k, v in corrupt_means.items()},
    "interaction": round(interaction, 3),
})
```

</details>

三个随机种子只能提供方差预警，不能给出精确的总体估计。应汇报各次运行、聚合方式与不确定性，而不是挑选最佳种子。如果研究者检查了大量变体，验证集也会变成优化反馈的一部分；只有在决策规则冻结之后，最终测试集才能被打开。负面消融也应进入研究日志，因为即使它没有改善头条分数，仍然能约束未来解释。


### **缩放定律与计算分配** {#scaling-laws-compute-allocation}

缩放定律是把损失与参数量 (N)、数据量 (D)、训练计算量 (C) 等资源连接起来的经验规律。一种常见的局部模型为

$$
L(C)=L_{\infty}+A C^{-\alpha},
$$

其中 (L_{\infty}) 是不可约或渐近误差下限，(A>0) 表示可降低损失的尺度，而 (alpha>0) 是减去下限后在双对数坐标上观察到的斜率。该公式不是关于所有架构或数据的定理，而是对已测量区间的拟合描述。

[Kaplan et al.](https://arxiv.org/abs/2001.08361) 记录了语言模型跨规模的幂律趋势；[Hoffmann et al.](https://arxiv.org/abs/2203.15556) 则说明在固定计算量下，模型大小和训练 token 数必须联合分配。更普遍的实验启示是：只比较参数量并不完整，在投入全部预算前，缩放先导实验应改变多个资源轴。

![实测先导点对局部插值的支持强于对远距离外推的支持。](assets/dl22-scaling-curve.svg){fig-align="center" width="74%" fig-alt="一条下降的损失曲线显示实测先导点，并用虚线标记不确定的外推区域。"}

<details>
<summary><strong>PyTorch：构建一个不冒充普遍定律的小型缩放先导实验</strong></summary>

```python
scaling_rows = []
for width in (16, 32, 64, 128):
    pilot_model, pilot_metrics = fit_model(
        seed=2210, width=width, dropout=0.1, epochs=7
    )
    parameters = sum(p.numel() for p in pilot_model.parameters())
    compute_proxy = parameters * len(train_ids) * 7
    scaling_rows.append({
        "width": width,
        "parameters": parameters,
        "compute_proxy": compute_proxy,
        "validation_nll": evaluate(pilot_model, val_ids)["nll"],
    })

# A two-parameter log-log fit is a diagnostic summary, not a forecast guarantee.
log_c = np.log([row["compute_proxy"] for row in scaling_rows])
log_l = np.log([row["validation_nll"] for row in scaling_rows])
slope, intercept = np.polyfit(log_c, log_l, deg=1)
assert all(np.isfinite([slope, intercept]))
print({"local_exponent": round(-float(slope), 3), "pilots": scaling_rows})
```

</details>

这个代理量忽略了反向传播常数、内存带宽、内核效率与超参数重调。严谨的缩放研究应记录实际见过的 token 或样本数、FLOPs 或加速器小时、墙钟时间、必要时的能源，以及推理成本。外推还应包含不确定性和停止规则：如果先导实验违反假设的单调区间，或预测收益小于测量噪声，就还没有理由继续扩大规模。


### **稀疏与自适应计算** {#sparse-adaptive-computation}

稠密网络对每个样本激活大部分参数。稀疏计算试图有选择地使用容量。**权重稀疏**移除单个连接，**结构化稀疏**移除块、通道、注意力头或层，**条件计算**则为不同输入选择不同路径。即使零元素数量相同，这些形式对硬件的影响也不同。

对于隐藏激活 (h\in\mathbb{R}^{D})，top-(k) 门控保留索引集合

$$
S_k(h)=\operatorname{TopK}(|h|,k),\qquad
\tilde h_i=h_i\,\mathbb{1}[i\in S_k(h)].
$$

这样可以把活跃值从 (D) 个降到 (k) 个，但稠密实现仍可能先计算全部 (D) 个激活再进行遮罩。真正加速需要内核和存储布局能够实际跳过这些工作。

![条件稀疏只让输入通过一部分计算路径。](assets/dl22-sparse-adaptive-computation.svg){fig-align="center" width="75%" fig-alt="输入进入 top-k 门控，激活两条计算路径，关闭另一条路径，最后合并结果。"}

<details>
<summary><strong>PyTorch：隔离 top-k 隐藏激活对准确率的影响</strong></summary>

```python
def logits_with_topk(model, x, k):
    model.eval()
    with torch.inference_mode():
        h1 = F.gelu(model.fc1(x))
        indices = h1.abs().topk(k, dim=1).indices
        mask = torch.zeros_like(h1).scatter_(1, indices, 1.0)
        sparse_h1 = h1 * mask
        h2 = F.gelu(model.fc2(sparse_h1))
        return model.out(h2), float(mask.mean())


sparse_results = {}
for k in (8, 16, 32, baseline_model.width):
    logits, active_fraction = logits_with_topk(baseline_model, images[test_ids], k)
    sparse_results[k] = {
        "active_fraction": active_fraction,
        "accuracy": float(logits.argmax(1).eq(targets[test_ids]).float().mean()),
    }

dense_logits = evaluate(baseline_model, test_ids)["logits"]
full_topk_logits, _ = logits_with_topk(
    baseline_model, images[test_ids], baseline_model.width
)
assert torch.allclose(dense_logits, full_topk_logits, atol=1e-6)
print({k: {m: round(v, 3) for m, v in row.items()} for k, row in sparse_results.items()})
```

</details>

这个实验是训练后的激活干预，而不是稀疏训练，也不是延迟基准。它回答一个狭窄的诊断问题：在预测发生变化前，可以移除该模型多少隐藏活动？部署研究还必须单独测量端到端延迟、内存流量、批处理行为、编译器支持，以及决定跳过哪些计算本身的成本。


### **混合专家与混合架构** {#mixture-experts-hybrid-architectures}

混合专家（Mixture-of-Experts, MoE）层包含多个参数化专家，但每个 token 或样本只激活其中一小部分。给定路由概率 (g_i(x)) 与专家输出 (E_i(x))，top-(k) 层计算

$$
y(x)=\sum_{i\in\operatorname{TopK}(g(x),k)}\hat g_i(x)E_i(x),
$$

其中 (hat g_i) 对被选中的权重重新归一化。总参数量可以随专家数增长，而活跃 FLOPs 更接近只含 (k) 个专家的稠密层。它的收益是条件容量，代价则包括路由复杂度、专家间通信、容量溢出，以及大量输入选择同一个专家时的不稳定性。

[Switch Transformer](https://www.jmlr.org/papers/v23/21-0998.html) 在大规模条件下展示了简化的 top-1 路由，并记录了通信和稳定性约束。混合架构组合具有互补性质的机制，例如 [Jamba](https://arxiv.org/abs/2403.19887) 交错使用注意力、状态空间与 MoE 组件。比较这类系统时，应同时考虑质量、活跃 FLOPs、内存占用、通信、吞吐量和长上下文行为，而不能只看参数量。

![混合块可以组合注意力、状态空间递归和稀疏专家。](assets/dl22-moe-hybrid.svg){fig-align="center" width="78%" fig-alt="Token 依次通过注意力、状态空间块和路由混合专家层。"}

<details>
<summary><strong>PyTorch：训练微型 top-1 MoE 并检查路由平衡</strong></summary>

```python
class TinyMoE(nn.Module):
    def __init__(self, width=48, experts=4):
        super().__init__()
        self.trunk = nn.Linear(64, width)
        self.router = nn.Linear(width, experts)
        self.experts = nn.ModuleList([nn.Linear(width, width) for _ in range(experts)])
        self.out = nn.Linear(width, 10)

    def forward(self, x):
        h = F.gelu(self.trunk(x))
        router_probs = self.router(h).softmax(dim=1)
        choices = router_probs.argmax(dim=1)
        all_outputs = torch.stack([F.gelu(expert(h)) for expert in self.experts], dim=1)
        chosen = all_outputs[torch.arange(len(x)), choices]
        return self.out(chosen), router_probs, choices


seed_everything(2220)
moe_model = TinyMoE()
moe_optimizer = torch.optim.AdamW(moe_model.parameters(), lr=2e-3)
for _ in range(9):
    moe_model.train()
    for x, y in make_loader(train_ids, shuffle=True, seed=2220):
        logits, router_probs, _ = moe_model(x)
        mean_load = router_probs.mean(dim=0)
        balance_loss = ((mean_load - 0.25) ** 2).mean()
        loss = F.cross_entropy(logits, y) + 0.2 * balance_loss
        moe_optimizer.zero_grad()
        loss.backward()
        moe_optimizer.step()

moe_model.eval()
with torch.inference_mode():
    moe_logits, _, moe_choices = moe_model(images[test_ids])
route_counts = torch.bincount(moe_choices, minlength=4)
moe_accuracy = float(moe_logits.argmax(1).eq(targets[test_ids]).float().mean())
assert int(route_counts.sum()) == len(test_ids)
print({"accuracy": round(moe_accuracy, 3), "routes": route_counts.tolist()})
```

</details>

辅助损失鼓励软路由概率分散，但平均负载平衡并不保证专家形成有意义的专门化。还要诊断 token 丢弃、不同领域的路由、路由熵、专家梯度和 all-to-all 通信。若批量很小或路由失衡，理论上活跃 FLOPs 很高效的 MoE 仍可能具有很差的墙钟利用率。


### **测试时计算** {#test-time-computation}

测试时计算用额外推理工作换取更好的决策。一般形式包括对多种变换进行集成、抽取多个候选输出、验证并选择候选、反复修订状态，或搜索动作树。在推理系统中，计算分配本身成为一种策略：简单输入应尽早停止，困难输入则获得更多采样或更深搜索。

令 (q(y\mid x,c)) 表示计算预算 (c) 下的解答质量。实际目标并非单纯最大化质量，而是选择

$$
c^*(x)=\arg\max_c\;\mathbb{E}[U(y,x)\mid c]-\lambda\,\operatorname{Cost}(c),
$$

其中 (U) 是任务效用，(lambda) 把延迟、金钱或能耗转换到同一决策尺度。[Snell et al.](https://arxiv.org/abs/2408.03314) 发现有效分配取决于问题难度和基础模型；这反驳了“总是增加采样”之类的普遍规则。

![控制器可以选择快速单次路径、重复采样或更深搜索。](assets/dl22-test-time-compute.svg){fig-align="center" width="76%" fig-alt="自适应控制器在生成答案前，把输入路由到单次推理、多次采样或搜索式推理。"}

<details>
<summary><strong>PyTorch：测量 Monte Carlo dropout 的计算阶梯</strong></summary>

```python
corrupt_x = images[test_ids].clone()
generator = torch.Generator().manual_seed(2230)
corrupt_x = (corrupt_x + 0.22 * torch.randn(corrupt_x.shape, generator=generator)).clamp(0, 1)


def mc_dropout_predict(model, x, passes):
    model.train()  # Dropout remains stochastic; this network has no batch normalization.
    samples = []
    with torch.inference_mode():
        for _ in range(passes):
            samples.append(model(x).softmax(dim=1))
    mean_probability = torch.stack(samples).mean(dim=0)
    entropy = -(mean_probability * mean_probability.clamp_min(1e-9).log()).sum(dim=1)
    return mean_probability, entropy


test_time_rows = []
for passes in (1, 4, 16):
    probability, entropy = mc_dropout_predict(baseline_model, corrupt_x, passes)
    accuracy = float(probability.argmax(1).eq(targets[test_ids]).float().mean())
    test_time_rows.append({
        "passes": passes,
        "accuracy": accuracy,
        "mean_entropy": float(entropy.mean()),
    })
baseline_model.eval()
assert [row["passes"] for row in test_time_rows] == [1, 4, 16]
print([{k: round(v, 3) if isinstance(v, float) else v for k, v in row.items()} for row in test_time_rows])
```

</details>

额外前向传播可能降低采样噪声，但准确率不必单调改善。有效研究应汇报完整的质量—成本曲线，计入验证器或控制器成本，并与使用相同预算的更强单次推理模型比较。搜索还会放大评估器弱点：针对不完善验证器进行优化，可能选出流畅但错误的候选。


### **具身学习与世界模型** {#embodied-learning-world-models}

具身学习把智能体放进反馈循环。智能体接收部分观测 (o_t)，选择动作 (a_t)，改变环境，并接收未来观测与奖励。与固定监督数据集不同，策略会影响之后产生什么数据。因此，探索、延迟后果、安全约束与分布偏移都是学习问题的核心组成部分。

世界模型把交互历史压缩为潜在状态 (z_t)，预测后果，并支持在想象轨迹中学习或规划：

$$
z_t\sim q_\phi(z_t\mid z_{t-1},a_{t-1},o_t),\qquad
(z_{t+1},r_t,\gamma_t)\sim p_\theta(\cdot\mid z_t,a_t).
$$

这里 (q_\phi) 是推断模型，(p_\theta) 预测潜在动力学、奖励与延续变量 (gamma_t)。策略可以从模拟 rollout 中改进，但只在模型已经学习到的支持范围内可靠。

![世界模型在观测、潜在动力学、想象 rollout、动作和现实校验之间形成闭环。](assets/dl22-world-model-loop.svg){fig-align="center" width="73%" fig-alt="观测被编码成潜在状态，经学习到的动力学展开，由策略评估后在现实中执行，并检查模型误差。"}

[DreamerV3](https://www.nature.com/articles/s41586-025-08744-2) 是通过潜在想象在多种控制领域学习行为的成熟案例。前沿范围更广：从视频和交互中学习、进行长程规划、跨身体与环境迁移，以及把生成式预测与可靠控制结合起来。未解问题并不只是视觉真实感。有用的世界模型必须表示受动作影响的因果关系、不确定性、罕见危险、对象持续性和干预后果。

研究主张应区分**预测质量**、**规划效用**和**现实有效性**。低重构误差可能忽略决策关键细节；视觉可信的 rollout 也可能违反动力学；智能体还可能利用模型错误。评估因此需要保留干预测试、长程误差累积测试、新颖条件下的不确定性，以及让想象中训练的策略进入物理系统之前的安全检查。第 17 章详细介绍核心强化学习和世界模型算法；本节关注它们的开放研究边界。


### **面向科学的深度学习** {#deep-learning-for-science}

科学深度学习把神经模型作为科学工作流中的工具。常见角色包括代理模拟、逆问题、参数估计、实验设计、结构预测、控制与假设生成。成功不能只由基准误差定义；预测还必须尊重单位、对称性、边界条件、守恒律、不确定性以及该领域真正采用的决策过程。

![科学深度学习从科学问题和领域知识出发，经学习组件到达领域验证，并形成反馈闭环。](assets/dl22-scientific-learning-stack.svg){fig-align="center" width="72%" fig-alt="四层结构连接科学问题、数据与控制知识、学习组件和领域验证，并从验证反馈到问题。"}

[AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2) 证明精心设计的神经表示与领域评估能够改变蛋白质结构预测。[GraphCast](https://doi.org/10.1126/science.adi2336) 从再分析数据中学习全球天气动力学，而 [NeuralGCM](https://www.nature.com/articles/s41586-024-07744-y) 把可微分大气动力学与学习组件结合。这些案例代表不同范式：数据驱动预测、学习型模拟器和物理—神经混合系统。它们都不意味着方程、实验或专家验证已经不再需要。

三个验证层次不可缺少：

| 层次 | 问题 | 典型证据 |
|---|---|---|
| 数值 | 模型能否在已测条件下近似目标？ | 保留误差、校准、守恒残差 |
| 科学 | 模型是否保留领域要求的机制与不变量？ | 干预测试、量纲检查、已知极限情形 |
| 操作 | 模型是否在成本与风险约束下改善真实工作流？ | 前瞻研究、专家比较、决策结果 |

最大的风险是从观测数据中发现捷径。模型可能利用模拟器伪影、数据同化惯例、实验批次或地理信息。科学主张需要跨仪器、机构、机制区间或时间进行外部验证，并报告模型在哪些地方超出了训练支持范围。混合模型可以编码可信结构，但错误约束可能比无约束近似更危险，因为它会制造虚假信心。


### **神经算子** {#neural-operators}

普通神经网络近似有限维向量之间的映射。神经算子试图学习函数之间的映射，例如从系数场 (a(x)) 到对应 PDE 解 (u(x))：

$$
\mathcal{G}_\theta:a(x)\mapsto u(x).
$$

[Fourier Neural Operator](https://arxiv.org/abs/2010.08895) 在频域中参数化全局混合。一个简化层为

$$
v_{t+1}(x)=\sigma\!\left(Wv_t(x)+\mathcal{F}^{-1}\!\left(R_\theta(k)\,\mathcal{F}(v_t)(k)\right)(x)\right),
$$

其中 (W) 是局部通道变换，(mathcal{F}) 是 Fourier 变换，(R_\theta(k)) 是对所选频率模式执行的可学习变换。由于学习参数作用于频率模式而非固定稠密网格，只要离散化与表示支持，同一个算子就可能在不同分辨率上求值。

![Fourier 神经算子把场变换到频域，学习选定模式，再返回空间网格。](assets/dl22-neural-operator.svg){fig-align="center" width="77%" fig-alt="一个场依次经过 FFT、可学习的保留频率模式、逆 FFT 和局部残差变换。"}

<details>
<summary><strong>PyTorch：验证频谱算子的形状与平移行为</strong></summary>

```python
def low_mode_operator(x, modes=3):
    # x has shape [B, H, W]; this fixed filter demonstrates the spectral path.
    spectrum = torch.fft.rfft2(x)
    filtered = torch.zeros_like(spectrum)
    filtered[:, :modes, :modes] = spectrum[:, :modes, :modes]
    filtered[:, -modes + 1 :, :modes] = spectrum[:, -modes + 1 :, :modes]
    return torch.fft.irfft2(filtered, s=x.shape[-2:])


digit_fields = images[test_ids[:16]].reshape(-1, 8, 8)
filtered_fields = low_mode_operator(digit_fields, modes=3)
shifted = torch.roll(digit_fields, shifts=(1, 2), dims=(-2, -1))
filtered_shifted = low_mode_operator(shifted, modes=3)
expected_shift = torch.roll(filtered_fields, shifts=(1, 2), dims=(-2, -1))
assert filtered_fields.shape == digit_fields.shape
assert torch.allclose(filtered_shifted, expected_shift, atol=1e-5)
print({"shape": tuple(filtered_fields.shape), "relative_error": float((filtered_fields-digit_fields).norm()/digit_fields.norm())})
```

</details>

这段代码是在数字图像上使用固定低通算子，而不是训练 PDE 求解器。它展示张量流和循环平移等变性。研究级神经算子还必须检验边界条件、混叠、网格几何、分辨率迁移、长程 rollout 稳定性、守恒性，以及在误差和计算量匹配时与数值求解器的比较。零样本超分辨率是需要逐问题验证的假设，而不是使用 FFT 自动获得的性质。


### **持续与终身学习** {#continual-lifelong-learning}

持续学习研究如何让系统从任务流或领域流中更新，同时避免使用全部历史数据重新训练。核心张力是**可塑性**与**稳定性**：参数既要吸收新信息，也要保留仍然有用的旧行为。当当前数据的更新覆盖了早期数据所需的表示时，就会发生灾难性遗忘。

必须准确命名实验设定。在**任务增量学习**中，推理时可以获得任务身份；在**领域增量学习**中，预测空间不变但输入分布改变；在**类别增量学习**中，新类别不断出现，模型必须在没有任务标签的情况下对所有类别进行分类。这些设定的难度和内存假设不同。

![持续学习器必须适应新任务，同时保留早期任务性能。](assets/dl22-continual-learning.svg){fig-align="center" width="74%" fig-alt="任务按顺序到达，评估覆盖所有任务，反馈弧表示 replay 或正则化帮助保留旧能力。"}

<details>
<summary><strong>PyTorch：在序列数字任务上测量遗忘与 replay</strong></summary>

```python
task_a_train = train_ids[np.isin(digits.target[train_ids], [0, 1, 2, 3, 4])]
task_b_train = train_ids[np.isin(digits.target[train_ids], [5, 6, 7, 8, 9])]
task_a_test = test_ids[np.isin(digits.target[test_ids], [0, 1, 2, 3, 4])]
task_b_test = test_ids[np.isin(digits.target[test_ids], [5, 6, 7, 8, 9])]


def continue_training(model, ids, seed, epochs=10, replay_ids=None):
    if replay_ids is not None:
        ids = np.concatenate([ids, replay_ids])
    optimizer = torch.optim.SGD(model.parameters(), lr=0.06, momentum=0.8)
    for epoch in range(epochs):
        model.train()
        for x, y in make_loader(ids, shuffle=True, seed=seed + epoch):
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model


seed_everything(2240)
naive_stream = DigitMLP(width=64, dropout=0.0)
continue_training(naive_stream, task_a_train, seed=2240, epochs=12)
a_before = evaluate(naive_stream, task_a_test)["accuracy"]
continue_training(naive_stream, task_b_train, seed=2250, epochs=12)
a_after = evaluate(naive_stream, task_a_test)["accuracy"]
b_after = evaluate(naive_stream, task_b_test)["accuracy"]

seed_everything(2240)
replay_stream = DigitMLP(width=64, dropout=0.0)
continue_training(replay_stream, task_a_train, seed=2240, epochs=12)
rng = np.random.default_rng(2240)
replay_ids = rng.choice(task_a_train, size=160, replace=False)
continue_training(
    replay_stream, task_b_train, seed=2250, epochs=12, replay_ids=replay_ids
)
replay_a = evaluate(replay_stream, task_a_test)["accuracy"]
replay_b = evaluate(replay_stream, task_b_test)["accuracy"]

assert set(targets[task_a_train].tolist()).isdisjoint(set(targets[task_b_train].tolist()))
print({
    "naive": {"A_before": round(a_before, 3), "A_after": round(a_after, 3), "B_after": round(b_after, 3)},
    "replay": {"A_after": round(replay_a, 3), "B_after": round(replay_b, 3)},
})
```

</details>

Replay 通常有效，但它会存储数据并改变采样分布。其他方法会约束重要参数、隔离任务专属容量、蒸馏旧行为或扩展模型。评估应报告已学习任务的平均准确率、遗忘、前向迁移、内存、更新计算量，以及是否提供任务边界或标签。通过存储全部数据并从头重训来保留所有知识，解决的是另一种资源问题。


### **因果表示学习** {#causal-representation-learning}

标准预测学习估计在训练分布中有用的关联。因果表示学习则询问：能否把高维观测映射为在干预和环境变化下仍具有明确意义的变量。在结构因果模型中，

$$
z_i=f_i(\operatorname{pa}(z_i),\epsilon_i),
$$

每个潜变量 (z_i) 由其父节点和独立扰动 (epsilon_i) 生成。只从像素或文本中学习这些变量很困难，因为许多潜在描述都能拟合同一个观测分布；可识别性需要额外假设、多环境、干预、时间结构或其他监督。

![稳定因果特征和依赖环境的捷径都可能在训练期间预测标签。](assets/dl22-causal-shift.svg){fig-align="center" width="72%" fig-alt="因果图显示环境同时影响数字内容和捷径，而两者都进入预测。"}

[Scholkopf et al.](https://arxiv.org/abs/2102.11107) 把因果表示与迁移和泛化联系起来：机制可以保持稳定，而干扰关联发生变化。这并不意味着不变性本身就能识别因果。一个恒定但错误的特征，也可能在研究者恰好收集的环境中表现为不变。

<details>
<summary><strong>Python：构造并诊断含虚假关联的训练环境</strong></summary>

```python
rng = np.random.default_rng(2260)
train_parity = digits.target[train_ids] % 2
test_parity = digits.target[test_ids] % 2

# The shortcut agrees with parity on 95% of training examples and is reversed
# at test time. Pixels remain the stable evidence source.
train_shortcut = np.where(rng.random(len(train_ids)) < 0.95, train_parity, 1 - train_parity)
test_shortcut = 1 - test_parity
stable_train = digits.data[train_ids] / 16.0
stable_test = digits.data[test_ids] / 16.0
spurious_train = np.column_stack([stable_train, 6.0 * train_shortcut])
spurious_test = np.column_stack([stable_test, 6.0 * test_shortcut])

stable_classifier = LogisticRegression(max_iter=500, random_state=2260).fit(
    stable_train, train_parity
)
spurious_classifier = LogisticRegression(max_iter=500, random_state=2260).fit(
    spurious_train, train_parity
)
stable_accuracy = accuracy_score(test_parity, stable_classifier.predict(stable_test))
shifted_accuracy = accuracy_score(test_parity, spurious_classifier.predict(spurious_test))
assert np.mean(train_shortcut == train_parity) > 0.90
assert np.mean(test_shortcut == test_parity) == 0.0
print({"stable_pixels": round(stable_accuracy, 3), "with_reversed_shortcut": round(shifted_accuracy, 3)})
```

</details>

这里的干预是人工构造的，像素模型也没有被证明是因果模型。它的作用是暴露一种可证伪失败：在一个环境中因高度可靠而被选中的预测因子，可能在数据收集机制变化后崩溃。更强证据需要收集多个真实环境，定义合理干预，在可识别时检验反事实预测，并声明哪些因果假设无法从数据中验证。


### **神经符号学习** {#neuro-symbolic-learning}

神经符号系统把学习到的感知或表示与逻辑规则、程序、知识图谱、类型或约束等显式结构组合起来。整合可以发生在多个位置：符号知识可以生成监督、约束损失、引导搜索、过滤解码输出，或在神经感知之后作为独立推理模块运行。

![神经感知产生不确定分数，符号层则执行显式有效性约束。](assets/dl22-neuro-symbolic-loop.svg){fig-align="center" width="74%" fig-alt="原始输入通过神经模型和符号约束层，约束还可以通过反馈路径影响学习。"}

它的吸引力来自互补优势。神经模型能够处理噪声和高维输入；符号系统可以表达组合规则，并产生可检查的证明步骤。接口同时也是主要弱点：离散决策会阻断梯度，神经不确定性可能过早丢失，规则库可能不完整，而错误硬约束可能强迫系统给出非常自信的错误输出。

<details>
<summary><strong>PyTorch：在数字对解码中使用奇偶约束</strong></summary>

```python
baseline_model.eval()
pair_ids = test_ids[:200]
pair_x = images[pair_ids]
pair_y = targets[pair_ids]
generator = torch.Generator().manual_seed(2270)
pair_x = (pair_x + 0.28 * torch.randn(pair_x.shape, generator=generator)).clamp(0, 1)
with torch.inference_mode():
    pair_probabilities = baseline_model(pair_x).softmax(dim=1)

direct_correct = 0
constrained_correct = 0
pair_count = len(pair_ids) // 2
for i in range(pair_count):
    p1, p2 = pair_probabilities[2 * i], pair_probabilities[2 * i + 1]
    y1, y2 = int(pair_y[2 * i]), int(pair_y[2 * i + 1])
    known_parity = (y1 + y2) % 2  # External symbolic side information.
    direct = (int(p1.argmax()), int(p2.argmax()))
    direct_correct += int(direct == (y1, y2))

    best_score, best_pair = -1.0, None
    for a in p1.topk(4).indices.tolist():
        for b in p2.topk(4).indices.tolist():
            if (a + b) % 2 == known_parity:
                score = float(p1[a] * p2[b])
                if score > best_score:
                    best_score, best_pair = score, (a, b)
    constrained_correct += int(best_pair == (y1, y2))

assert constrained_correct >= direct_correct
print({"pairs": pair_count, "direct_correct": direct_correct, "constrained_correct": constrained_correct})
```

</details>

该约束不会破坏已经正确的数字对，因为真实数字对必然满足它，但它使用了普通识别任务中无法获得的额外奇偶信息。因此，这个比较展示的是约束解码，而不是免费提升准确率。研究必须计入获取符号知识的成本，检验规则违反与冲突，在接口间保留不确定性，并与获得等价额外信息的神经模型比较。


### **选择开放研究问题** {#choosing-open-research-problem}

一个开放主题还不是研究问题。“研究世界模型”只命名了领域；“在固定交互预算下，判断由不确定性门控的规划能否减少不安全的模型漏洞利用”才提出了可证伪关系。好的问题把重要性、未解机制、可行证据和实验后可能改变的决策连接起来。

![连续施加约束可以把广泛的重要领域转化为可研究决策。](assets/dl22-problem-funnel.svg){fig-align="center" width="68%" fig-alt="四阶段漏斗从重要问题收窄为可证伪问题、可行证据和决策规则。"}

可以使用五个筛选条件：

1. **重要性：**如果问题得到回答，谁或什么会发生改变？错误答案的代价是什么？
2. **缺口：**不确定性属于科学、经验、工程还是评估问题？基准表中少一项结果并不自动构成有意义的缺口。
3. **可处理性：**能否利用可获得的数据、计算、专长和时间运行决定性实验？
4. **可识别性：**所提证据能否把目标机制与混杂因素和竞争解释区分开？
5. **贡献路径：**即使结果为负，能否仍然产出有用的数据集、协议、诊断、边界或设计规则？

<details>
<summary><strong>Python：把研究问题选择标准显式化</strong></summary>

```python
candidates = {
    "adaptive_test_time_compute": {
        "importance": 4, "gap": 4, "tractability": 5, "identifiability": 4, "negative_value": 5
    },
    "moe_routing_balance": {
        "importance": 4, "gap": 3, "tractability": 4, "identifiability": 4, "negative_value": 4
    },
    "causal_digit_representation": {
        "importance": 5, "gap": 5, "tractability": 2, "identifiability": 1, "negative_value": 3
    },
}
weights = {
    "importance": 0.25,
    "gap": 0.20,
    "tractability": 0.20,
    "identifiability": 0.25,
    "negative_value": 0.10,
}
scores = {
    name: sum(criteria[key] * weight for key, weight in weights.items())
    for name, criteria in candidates.items()
}
ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
assert ranking[0][0] == "adaptive_test_time_compute"
print(ranking)
```

</details>

这些数字不会让决策变得客观，它们只是暴露决策理由。敏感性分析应改变权重，领域专家也应挑战重要性与可识别性得分。一个小而有决定力的实验，通常比一个只能用低功效代理进行检验的宏大问题更适合作为起点。


### **维护研究日志** {#maintaining-research-log}

研究日志应在替代方案仍然清晰时记录决策。记录单位不能只是“今天做了模型”，而应是一个可重建事件：问题、假设、相对上次运行的改变、配置、数据版本、环境、结果、解释和下一步决策。只记录成功运行会制造后见之明偏差，并让后来的人付出高昂成本去重新发现失败方向。

![研究日志把问题、配置、数据、运行、指标、决策和产物连接起来。](assets/dl22-research-log.svg){fig-align="center" width="76%" fig-alt="沿袭图把研究问题连接到配置、数据、运行、指标、决策和保存产物，负面结果再反馈到下一个问题。"}

应把不可变事实与解释分开。配置、校验和、原始预测、时间戳和环境元数据应由机器生成；假设、意外结果和停止决策则需要人工解释。保存精确命令，并按内容哈希或版本化路径保存生成产物；一个可被覆盖的 `final_results.csv` 文件不构成沿袭信息。

<details>
<summary><strong>Python：生成按内容寻址的实验记录</strong></summary>

```python
run_config = {
    "model": "DigitMLP",
    "width": 64,
    "dropout": 0.10,
    "epochs": 12,
    "optimizer": "AdamW",
    "learning_rate": 2e-3,
    "seed": 2201,
}
config_json = json.dumps(run_config, sort_keys=True, separators=(",", ":"))
run_id = hashlib.sha256(config_json.encode("utf-8")).hexdigest()[:12]
research_record = {
    "run_id": run_id,
    "question": claim_card["question"],
    "config": run_config,
    "dataset_sha256": dataset_digest,
    "split_sha256": manifest["split_sha256"],
    "metrics": {k: round(v, 6) for k, v in baseline_metrics.items()},
    "interpretation": "Baseline for controlled width/dropout ablations.",
    "decision": "retain as reference; do not tune on test metrics",
}
record_json = json.dumps(research_record, indent=2, sort_keys=True)
assert research_record["run_id"] == hashlib.sha256(config_json.encode()).hexdigest()[:12]
assert research_record["dataset_sha256"] == manifest["dataset_sha256"]
print(record_json)
```

</details>

成熟项目还应记录伦理与治理决策、许可证、访问限制、资源消耗、审稿反馈和对原计划的偏离。[Turing Way](https://book.the-turing-way.org/) 提供了一本采用 CC BY 4.0 许可、面向可复现、合乎伦理且协作式研究的开放手册。日志的目的不是增加官僚流程，而是充分保留项目的因果历史，使其他人——也包括未来的作者自己——能够理解某个产物为何存在。


### **章节对比与总结** {#chapter-comparison-summary}

研究实践与前沿认知解决不同问题。研究实践决定证据是否可信，前沿地图决定重要不确定性位于何处。用混杂基线研究的新架构属于薄弱研究；在无关问题上完成的完美可复现实验，也只是组织得很好的工作。高质量研究必须同时对齐二者。

| 领域 | 核心研究问题 | 必要证据 | 常见失败 |
|---|---|---|---|
| 复现与基线 | 结果能否重新生成并被公平比较？ | 研究产物、匹配预算、独立检查 | 把一次确定性重跑当作普遍重复验证 |
| 消融 | 哪个干预导致变化？ | 受控析因比较、多随机种子 | 同时改变容量、调参和数据 |
| 缩放 | 应如何分配有限资源？ | 多轴先导实验、不确定性、成本统计 | 从狭窄区间进行远距离外推 |
| 稀疏与混合模型 | 容量能否增长而活跃计算不按比例增长？ | 质量、路由、通信、延迟、内存 | 把理论 FLOPs 等同于系统速度 |
| 测试时计算 | 哪些输入能从更多推理工作中受益？ | 成本匹配曲线与验证器分析 | 忽略选择和验证成本 |
| 世界模型 | 学习到的动力学能否支持安全规划？ | 干预和长程检验 | 用视觉可信度判断实用性 |
| 科学学习 | 模型能否支持有效科学工作流？ | 领域约束、不确定性、外部验证 | 用基准误差代替科学有效性 |
| 神经算子 | 函数到函数映射能否跨离散化迁移？ | 边界、网格、rollout 与求解器比较 | 假定使用 FFT 就保证物理保真度 |
| 持续学习 | 系统能否在保留旧能力的同时适应？ | 遗忘、迁移、内存、更新成本 | 隐藏任务标签或历史数据假设 |
| 因果表示 | 哪些学习因素在干预下仍有意义？ | 显式假设、多环境、干预 | 把相关性或不变性称为因果 |
| 神经符号学习 | 学习到的不确定性应如何与显式规则交互？ | 接口测试、规则失败、等价额外信息 | 把特权约束当作免费准确率 |

本章实验形成一条紧凑证据链：固定 UCI 划分、已声明问题、可重复基线、析因消融、缩放先导实验、稀疏与路由变体、测试时计算曲线、序列任务、受控捷径偏移、约束解码和按内容寻址的日志。每个结果都被明确限制为机制研究。真正可迁移的技能正是这种纪律：询问改变了什么、固定了什么、哪种证据会推翻结论，以及当前实验无法证明什么。

随着证据变化，新兴前沿也应被重新审视。稀疏专家、混合序列模型、测试时计算、世界模型、科学学习、神经算子、持续适应、因果表示和神经符号整合，并不是通向“通用智能”的单一路线。它们分别回应容量分配、推理、交互、科学结构、适应和推理能力的不同限制。最持久的研究贡献可能是模型，也可能是更清晰的问题、可靠评估协议、负面结果、沿袭更完善的数据集，或防止一个诱人主张被推广过远的边界。
